In [37]:
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

In [38]:
df_films = pd.read_csv('../data/limpos/pixar_films.csv')
df_rating = pd.read_csv('../data/limpos/public_response.csv')
df_box = pd.read_csv('../data/limpos/box_office.csv')
df_genre = pd.read_csv('../data/limpos/genres.csv')
df_academy = pd.read_csv('../data/limpos/academy.csv')
df = pd.read_csv('../data/limpos/df.csv')

1. Como a bilheteria e a avaliação se comportam ao decorrer dos anos?

In [39]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=df["release_date"], y=df["imdb_score"], mode="lines+markers", name="Avaliação IMDb"), secondary_y=False)

fig.add_trace(go.Scatter(x=df["release_date"], y=df["box_office_worldwide"], mode="lines+markers", name="Bilheteria Mundial"), secondary_y=True)

fig.update_layout(template="plotly_dark")

fig.update_yaxes(title_text="Avaliação IMDb", secondary_y=False)

fig.update_yaxes(title_text="Bilheteria mundial", secondary_y=True)

fig.write_image('../imgs/01.Avaliacao_Bilheteria_Ano.png')

fig.show()

O gráfico demonstra uma baixa correlação direta entre a aprovação no IMDb e o sucesso de arrecadação mundial, mostrando que picos de bilheteria nem sempre acompanham as notas mais altas do público e vice-versa.

In [40]:
df_genre = pd.merge(df_rating, df_genre,on='film', how='left')
df_genre = pd.merge(df_box, df_genre,on='film', how='left')
df_genre2 = df_genre
df_genre = df_genre.groupby('value', as_index=False)['imdb_score'].mean()
df_genre = df_genre.sort_values(by='imdb_score', ascending=True)
print(df_genre)


                      value  imdb_score
37              Time Travel    6.100000
11                    Crime    6.200000
30                      Spy    6.200000
7                Car Action    6.450000
20               Motorsport    6.700000
29                    Sport    6.700000
12       Dinosaur Adventure    6.700000
25                  Romance    7.000000
0                    Action    7.080000
34           Teen Adventure    7.100000
31                Superhero    7.200000
15               Fairy Tale    7.250000
28             Space Sci-Fi    7.250000
33          Sword & Sorcery    7.250000
35              Teen Comedy    7.300000
26                   Sci-Fi    7.333333
18             Fantasy Epic    7.400000
3          Animal Adventure    7.480000
4                 Animation    7.542857
1                 Adventure    7.542857
10       Computer Animation    7.542857
23                    Quest    7.566667
8                    Comedy    7.571429
9             Coming-of-Age    7.583333


2. Como o gênero dos filmes influenciam  na opinião pública?

In [41]:
fig2 = px.bar(df_genre, x='value', y='imdb_score', template='plotly_dark', labels={'value': 'Gênero', 'imdb_score': 'Avaliação IMDb'}, color_continuous_scale='Blues', color='imdb_score', title='Rating por gênero')
fig2.write_image('../imgs/02.Rating_Genero.png')
fig2.show()

O gráfico apresenta a média de avaliação no IMDb por gênero cinematográfico, organizada em ordem crescente. Enquanto subgêneros como Time Travel, Crime e Spy registram as menores pontuações (ao redor de 6,0), categorias como Drama, Adventure Epic, Artificial Intelligence e Dystopian Sci-Fi lideram o ranking com as maiores notas (acima de 8,0).

In [42]:
def score(status):
    if status == 'Won':
        return 2
    if status == 'Nominated':
        return 1
    return 0

df_academy['score'] = df_academy['status'].apply(score)

df_agrup = df_academy.groupby('film', as_index=False)['score'].sum()

df_agrup = pd.merge(df_agrup, df, how='left')

df_agrup = df_agrup.groupby('franchise', as_index=False)['score'].sum()
        
df_agrup = df_agrup.sort_values(by='score', ascending=False)

3. Prêmios e indicações por franquia de filme. A franquia influencia na opinião dos críticos?

In [43]:
fig3 = px.bar(df_agrup, x='score', y='franchise', template='plotly_dark', color='franchise', title='Prêmios ou indicações por franquia', labels={'franchise': 'Franquia',
                                                                                                                                                  'score': 'Indicações'})
fig3.write_image('../imgs/03.Premios_Franquia.png')
fig3.show()

O gráfico destaca a liderança da franquia Toy Story em número de prêmios ou indicações (7), seguida por Divertidamente e Os incríveis (4 cada), enquanto Carros ocupa a última posição com apenas 1 indicação.

4. Existe correlação entre bilheteria e a opinião pública?

In [44]:
fig4 = px.scatter(df, x='imdb_score', y='box_office_worldwide', color='film', template='plotly_dark', title='Bilheteria x Score', labels={'box_office_worldwide': 'Bilheteria Mundial',
                                                                                                                                          'imdb_score': 'Avaliação IMDB', 'film': 'Filme'})
fig4.write_image('../imgs/04.Bilheteria_Score.png')
fig4.show()

O gráfico de dispersão mostra uma leve tendência positiva entre a nota no IMDb e a arrecadação mundial dos filmes. Embora as produções com maior faturamento apresentem avaliações elevadas (acima de 7,5), a dispersão dos pontos indica que uma nota alta no IMDb não garante, por si só, o sucesso de bilheteria.

5. Quais são os filmes com maior lucro?

In [45]:
df_agrup = df.groupby('film', as_index=False)[['box_office_worldwide', 'budget']].sum()

df_agrup['profit'] = df_agrup['box_office_worldwide'] - df_agrup['budget']
df_agrup['profit'] = df_agrup['profit'].astype(int)
df_agrup = df_agrup.sort_values(by='profit', ascending=False)

fig5 = px.histogram(df_agrup, x='film', y='profit', color='film', template='plotly_dark', labels={'profit': 'Lucro',
                                                                                                  'film': 'Filme'}, title='Lucro por filme')
fig5.write_image('../imgs/05.Lucro_Filme.png')
fig5.show()

O gráfico aponta Divertidamente 2 como o filme mais lucrativo do catálogo (próximo a 1,5 bilhão em lucro), seguido por grandes sequências como Incríveis 2 e Toy Story 4, enquanto produções como Luca e Red: Crescer é uma Fera figuram no extremo oposto com resultados negativos.

In [46]:
df_agrup = df_genre2.groupby('value', as_index=False)[['box_office_worldwide', 'imdb_score']].mean()
df_agrup['box_office_worldwide'] = df_agrup['box_office_worldwide'].astype(int)
df_agrup


,value,box_office_worldwide,imdb_score
0,Action,594372749,7.080000
1,Adventure,608664319,7.542857
2,Adventure Epic,521311860,8.400000
3,Animal Adventure,643755696,7.480000
4,Animation,608664319,7.542857
5,Artificial Intelligence,521311860,8.400000
6,Buddy Comedy,531608121,7.825000
7,Car Action,471891526,6.450000
8,Comedy,606418293,7.571429
9,Coming-of-Age,650441683,7.583333


6. Como é a relação entre orçamento e bilheteria?

In [47]:
fig = px.scatter(
    df,
    x='budget',
    y='box_office_worldwide',
    template='plotly_dark',
    title='Orçamento x Bilheteria dos Filmes da Pixar',
    labels={
        'budget': 'Orçamento',
        'box_office_worldwide': 'Bilheteria Mundial'
    },
    hover_name='film',
    color='film'
)

fig.update_xaxes(
    range=[0, 220_000_000],
    tickformat='$,.0f',
    dtick=25_000_000
)

fig.write_image('../imgs/06.Orcamento_Bilheteria.png')
fig.show()

O gráfico indica que orçamentos elevados (em torno de US$ 200M) viabilizam os maiores picos de bilheteria da Pixar, ultrapassando a marca de US$ 1,5B. No entanto, um grande investimento não garante o sucesso financeiro por si só, visto que filmes com orçamentos semelhantes apresentaram retornos expressivamente baixos ou próximos de zero.

In [48]:
df

,number,film,release_date,imdb_score,budget,box_office_worldwide,franchise
0,1,Toy Story,1995-11-22,8.3,30000000.0,394436586,NaN
1,2,A Bug's Life,1998-11-25,7.2,120000000.0,363258859,NaN
2,3,Toy Story 2,1999-11-24,7.9,90000000.0,511358276,Toy Story
3,4,"Monsters, Inc.",2001-11-02,8.1,115000000.0,528773250,"Monsters, Inc. & University"
4,5,Finding Nemo,2003-05-30,8.2,94000000.0,871014978,Finding Nemo/Dory
5,6,The Incredibles,2004-11-05,8.0,92000000.0,631442092,The Incredibles
6,7,Cars,2006-06-09,7.2,120000000.0,461983149,Cars
7,8,Ratatouille,2007-06-29,8.1,150000000.0,623726085,NaN
8,9,WALL-E,2008-06-27,8.4,180000000.0,521311860,NaN
9,10,Up,2009-05-29,8.3,175000000.0,735099082,NaN


7. Como as franquias se saem na bilheteria?

In [49]:
df_sunburst = df.copy()

df_sunburst['franchise'] = df_sunburst['franchise'].fillna('Standalone')

fig = px.sunburst(
    df_sunburst,
    path=['franchise', 'film'],
    values='box_office_worldwide',
    color='imdb_score',
    color_continuous_scale='Blues',
    template='plotly_dark',
    title='Bilheteria dos Filmes por Franquia'
)
fig.write_image('../imgs/07.Bilheteria_Franquia.png')
fig.show()

O gráfico sunburst ilustra a distribuição da bilheteira global por franquia (anel interno) e por filme (anel externo), destacando o grande peso comercial de Toy Story, Divertidamente e dos títulos Standalone, com os tons de azul mais escuros a assinalar as melhores notas no IMDb.

8. As franquias representam quanto da bilheteria mundial?

In [50]:
# df_donut = df_sunburst[
#     df_sunburst['franchise'] != 'Standalone'
# ]

df_sunburst = (
    df_sunburst
    .groupby('franchise', as_index=False)['box_office_worldwide']
    .sum()
)

fig = px.pie(
    df_sunburst,
    names='franchise',
    values='box_office_worldwide',
    hole=0.5,
    template='plotly_dark',
    title='Participação das Franquias na Bilheteria Mundial'
)

fig.update_traces(
    textinfo='percent+label'
)
fig.write_image('../imgs/08.Participacao_Franquia.png')
fig.show()

O gráfico de rosca mostra que os filmes sem franquia (Standalone) representam a maior quota da bilheteira mundial (30,3%), enquanto Toy Story (16,9%) e Divertidamente (15%) se destacam como as franquias de maior impacto comercial.

9. Quantos filmes são bem avaliados?

In [52]:
df2 = pd.merge(df, df_rating, on='film', how='left')

fig = px.histogram(
    df2,
    x='rotten_tomatoes_score',
    template='plotly_dark',
    title='Distribuição das Notas Rotten Tomatoes'
)

fig.update_traces(
    xbins=dict(
        start=40,
        end=100,
        size=10
    )
)

fig.update_layout(
    xaxis=dict(
        title='Nota Rotten Tomatoes',
        tick0=40,
        dtick=10,
        range=[40, 100]
    ),
    yaxis_title='Quantidade de Filmes',
    bargap=0.05
)
fig.write_image('../imgs/09.Dist_Notas.png')
fig.show()

O histograma demonstra uma forte concentração de aclamação crítica, com a maioria esmagadora dos filmes a obter notas entre 90% e 100% no Rotten Tomatoes. Em contrapartida, raríssimas produções registaram pontuações abaixo dos 70%, evidenciando o elevado padrão de aprovação dos projetos.